# นับดอกไม้ + บอกชนิด จากหลายรูป (เดโม 4 ชนิด v5 · ตัวที่ใช้นำเสนอ + เว็บ Bloom Craft + เจนช่อดอกไม้)

> **เดโม 4 ชนิด v3 (SAM 3 + ตัวบอกชนิดที่เราเทรน v2: other มี Oxford 102)** 2026-11-14 — เวอร์ชันเก่า: old_versions/flower_counter_demo_4ชนิด_v1_zeroshot.ipynb, _v2_head_dandelion.ipynb
> — นับเฉพาะ rose / tulip / sunflower / daisy (ดอกชนิดอื่น = unknown)
> สีดอกคิดจากรูปทรงดอกจริง (mask ของ SAM 3) และเซลล์สุดท้ายส่งออก `results/bouquet_input/` (JSON + รูปดอกตัดพื้นหลัง) ให้ขั้นเจนช่อดอกไม้

> **เวอร์ชัน v5.0** (2026-11-14) — ตัวหาดอกเปลี่ยนเป็น **SAM 3** (`DETECTOR = "sam3"`) ส่วนบอกชนิด/ด่านกรองเหมือน v4.9
> - SAM 3 เป็นโมเดลที่ต้องขอสิทธิ์: กด Request access ที่ https://huggingface.co/facebook/sam3
>   Colab: ใส่ token ใน Secret ชื่อ `HF_TOKEN` (เปิด Notebook access) | เครื่องตัวเอง: `hf auth login`
> - ใช้การ์ดจอ ~3GB (fp16) — ไม่มี GPU ช้ามาก, กลับไปใช้ `DETECTOR = "owl2"` ได้

> **เวอร์ชัน v4.9** (2026-10-20) — เวอร์ชันเก่าอยู่ในโฟลเดอร์ `version/`

```
เพิ่มรูป (หลายรูป/หลายรอบ) → กด "ตรวจทั้งหมด"
   แต่ละรูป: ด่าน 1 เป็นดอกไม้ไหม? (CLIP) → ไม่ใช่ = แจ้งเตือน
             ด่าน 2 เป็นภาพถ่ายจริงไหม? การ์ตูน/เกม/ภาพวาด = แจ้งเตือน, ดอกไม้ปลอม (พลาสติก/ผ้า) = ผ่าน
             ใช่ → detect ทีละดอก (OWLv2 หรือ YOLO) → crop → ตัดของที่ไม่ใช่ดอกไม้
                 → BioCLIP 2 ทายชนิดทีละดอก (ชื่อวิทยาศาสตร์ + รูปตัวอย่าง + บริบททั้งรูป)
   สุดท้าย: สรุปรวมทุกรูป (ดอกทั้งหมดกี่ดอก ชนิดละกี่ดอก อยู่ในกี่รูป)
```

**สิ่งที่เปลี่ยนจากเวอร์ชันเดิม**
1. prompt ของ BioCLIP ใช้ **ชื่อวิทยาศาสตร์** (เช่น *Tulipa gesneriana*) เพราะ BioCLIP เทรนด้วยชื่อแบบนี้
2. อัปเกรดเป็น **BioCLIP 2** (ทายชนิดแม่นกว่ารุ่นแรกมาก)
3. ผสม **บริบทของทั้งรูป** เข้ากับแต่ละดอก (`CONTEXT_WEIGHT`)
4. ใส่ **รูปตัวอย่าง** ของแต่ละชนิดได้ (`refs/<ชนิด>/*.jpg`) แม่นที่สุดในบรรดาทุกข้อ
5. Detector เริ่มต้นเป็น **OWLv2** (zero-shot หาดอกไม้ได้ทุกแบบ) หรือ YOLO ที่ train แบบ `single_cls`
6. ตัดกล่องที่ครอบทั้งช่อออก ไม่ให้นับเป็น 1 ดอก
7. **(v4)** ปฏิเสธดอกไม้การ์ตูน/ในเกม และดอกไม้ปลอมให้ใช้ CLIP ViT-L-14 ช่วยบอกชนิด (โหลดเฉพาะตอนเจอดอกไม้ปลอม)
8. **(v4.1) นับทีละดอกแม่นขึ้น** ทดสอบกับ 15 รูป ผิดรวมจาก ~65 ดอก เหลือ ~10 ดอก
   - OWLv2 ค้นด้วย `"flower"` + ชื่อดอกไม้ทุกชนิดใน `flowers` (เดิม `"a photo of a flower"` ได้กล่องครอบทั้งรูป แล้วกล่องดอกจริงถูกตัดทิ้ง)
   - เกณฑ์คะแนนปรับตามรูป (`DET_REL` × คะแนนสูงสุดในรูป) เพราะคะแนน OWLv2 ต่างกันมากในแต่ละรูป
   - กล่องซ้อนกัน (กลีบในดอก / ดอก+ก้าน) เหลือกล่องเดียว และเลือกกล่องที่เป็นตัวดอก
   - ภาพถ่ายจริงพื้นดำไม่ถูกปัดเป็นการ์ตูนแล้ว และดอกจริงไม่ถูกเรียกว่าดอกไม้ปลอม (`DRAWN_THR`, `ARTIFICIAL_THR` = 0.8)
9. **(v4.2) YOLO นับแม่นขึ้น**
   - `YOLO_CONF` 0.25 → 0.15 และเปิด TTA (`YOLO_TTA`) หาดอกที่เคยหลุด (ดอกด้านหลัง/ชนิดที่ไม่มีใน dataset)
   - ตัดกล่องเล็กผิดปกติ (`MIN_REL_AREA` × ขนาดกลางของกล่องในรูป) = เศษกลีบ ไม่ใช่ดอก
   - ด่านกรอง crop รู้จักริบบิ้น/กระดาษห่อ ไม่นับเป็นดอก
10. **(v4.3) รันใน VS Code ได้** อ่านรูปจากโฟลเดอร์ (`IMAGE_DIR`) แทนการอัปโหลด, รูปตัวอย่างวางใน `refs/<ชนิด>/`
11. **(v4.4) เลือก detector ได้ 3 แบบ** `"yolo"`, `"owl2"`, `"yolo+owl2"` (YOLO เป็นหลัก + เติมดอกที่ OWLv2 เจอแต่ YOLO ไม่เจอ)
   - แก้บั๊ก: ดอกริมรูปที่มีเศษกลีบซ้อนอยู่ถูกมองเป็นกล่องทั้งช่อแล้วถูกตัดทิ้ง (ตัดเศษกลีบก่อนแล้ว)
12. **(v4.5) ช่อดอกไม้ผสมหลายชนิด + เซลล์สรุปไม่ค้าง**
   - ใช้บริบททั้งรูปเฉพาะรูปที่เป็นดอกชนิดเดียวชัดๆ (`CONTEXT_MIN`) ช่อผสมเคยถูกดึงไปเป็นชนิดเดียวกันหมด/กลายเป็น unknown
   - `CROP_FLOWER_THR` 0.25 → 0.5 ตัดกล่องแจกัน/ใบไม้/กิ่ง (ได้ 0.03–0.35 ส่วนดอกจริงได้ ≥ 0.54)
   - โหลด CLIP ของดอกไม้ปลอมไว้ตั้งแต่เซลล์โหลดโมเดล (เดิมไปดาวน์โหลด 1.7GB กลางเซลล์สรุป ดูเหมือนค้าง) + บอกความคืบหน้าทีละรูป
13. **(v4.6)** `yolo+owl2`: ดอกใหญ่ที่ OWLv2 เจอแต่มีกล่องเล็กของ YOLO (เช่น ยิปโซ) อยู่ข้างใน ไม่ถูกตัดทิ้งแล้ว (ทานตะวันในช่อผสมเคยหาย)
   - เพิ่มชนิด goldenrod, scabiosa, clematis, craspedia (เจอบ่อยในช่อผสม เดิมขึ้น unknown)
14. **(v4.7)** `yolo+owl2`: กล่อง OWLv2 ที่ไม่มั่นใจ (เช่น กล่องแจกันที่มีทิวลิปอยู่ด้านบน) ไม่แทนกล่อง YOLO แล้ว
15. **(v4.8) ดอกช่อ** (ดอกเข็ม ไฮเดรนเยีย snapdragon ฯลฯ) นับ 1 ช่อ = 1 ดอกได้ (`COUNT_CLUSTER_AS_ONE`) เดิมกล่องทั้งช่อถูกตัดทิ้ง เหลือแต่ดอกย่อยกระจัดกระจาย
   - **บอกสีของแต่ละดอก** (RGB + ชื่อสี) ตัดพื้นหลังสีเขียว/มืดออกก่อน แล้วใช้ค่ากลางของพิกเซลที่เหลือ
   - เซลล์โหลดโมเดลดาวน์โหลด CLIP ดอกไม้ปลอมลงดิสก์อย่างเดียว ไม่โหลดเข้า RAM (ประหยัด ~1.7GB ตอนโหลด) และข้ามได้ถ้า `ARTIFICIAL_CLIP_WEIGHT = 0`
   - `CONTEXT_WEIGHT` 0.3 → 0.5 (ใช้เฉพาะรูปชนิดเดียวอยู่แล้ว) ดอกย่อยของดอกเข็มเคยถูกทายเป็นลีลาวดี
16. **(v4.9)** เปลี่ยน YOLO เป็น **YOLO11s** (`runs/detect/flower_single_yolo11/`) ใช้ dataset เดิม train แบบคลาสเดียวเหมือนเดิม

รันทีละเซลล์จากบนลงล่าง — **Colab**: Runtime → GPU | **VS Code**: เลือก kernel Python 3.10 แล้วรันได้เลย (รูปอ่านจาก `IMAGE_DIR`, รูปตัวอย่างวางใน `refs/<ชนิด>/`, YOLO ต้องมี `runs/detect/flower_single/weights/best.pt`)

## 1) ติดตั้ง

In [ ]:
%pip install -q open_clip_torch transformers

In [ ]:
# (Colab) ดึงโค้ด + ส่วนหัวที่เราเทรน + รูปเดโม + เว็บ จาก GitHub · เครื่องตัวเองข้ามเซลล์นี้เอง
import os, sys
HEAD_FILE = "runs/classify/species_bioclip_head_v2.pt"
if "google.colab" in sys.modules and not os.path.exists(HEAD_FILE):
    if not os.path.exists(f"/content/flower/{HEAD_FILE}"):   # ยังไม่มี หรือโหลดค้างจากรอบก่อน → โหลดใหม่
        print("กำลังดาวน์โหลดจาก GitHub ราว 150MB ใช้เวลา 1-3 นาที ห้ามกดหยุด ...", flush=True)
        !rm -rf /content/flower
        !git clone --depth 1 https://github.com/039-Nippitch/flower /content/flower
    else:   # มีอยู่แล้ว (runtime เดิม) → อัปเดตเว็บ/โค้ดเป็นตัวล่าสุด
        !git -C /content/flower pull -q --ff-only
    assert os.path.exists(f"/content/flower/{HEAD_FILE}"), "ดาวน์โหลดจาก GitHub ไม่สำเร็จ: เช็กเน็ต แล้วรันเซลล์นี้ใหม่"
    os.chdir("/content/flower")
print("folder:", os.getcwd())


In [ ]:
# (ไม่บังคับ) รันเซลล์นี้เฉพาะถ้าจะใช้ DETECTOR = "yolo" หรือ "yolo+owl2" — ถ้าใช้ "owl2" ข้ามได้เลย
%pip install -q roboflow ultralytics
import os, torch

# ตั้งค่า YOLO11 (รุ่นเดิม: YOLO_RUN = "flower_single", YOLO_BASE = "yolov8s.pt", รูปแบบ dataset "yolov8")
YOLO_RUN = "flower_single_yolo11"
YOLO_BASE = "yolo11s.pt"
if os.path.exists(f"runs/detect/{YOLO_RUN}/weights/best.pt"):
    print(f"มีโมเดลที่ train แล้ว ข้ามการ train (ลบโฟลเดอร์ runs/detect/{YOLO_RUN} ถ้าจะ train ใหม่)")
elif not torch.cuda.is_available():   # train บน CPU ใช้หลายชั่วโมง
    print(f"⚠️ ไม่มี GPU: ดาวน์โหลด best.pt จาก Colab มาวางที่ runs/detect/{YOLO_RUN}/weights/best.pt")
else:
    from roboflow import Roboflow
    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    project = rf.workspace("elbrusds-lzhr7").project("bouquet")
    dataset = project.version(8).download("yolov11")

    from ultralytics import YOLO
    model = YOLO(YOLO_BASE)   # s ใหญ่กว่า n นิดหน่อย แต่หาดอกได้ครบกว่า
    # single_cls=True → ให้ YOLO ทำหน้าที่ "หาว่าดอกไม้อยู่ตรงไหน" อย่างเดียว ส่วนการบอกชนิดให้ BioCLIP ทำ
    model.train(data=f"{dataset.location}/data.yaml", epochs=60, imgsz=640,
                single_cls=True, name=YOLO_RUN, exist_ok=True)

## 2) ตั้งค่า

In [ ]:
# ---------- ตัวตรวจจับ (หา "ตำแหน่ง" ดอกไม้) ----------
DETECTOR = "sam3"   # v5.0: sam3 ผิดรวมน้อยกว่า owl2 ใน 23 รูปทดสอบ | เดิม: ส่งงาน 9 พ.ย.: owl2 นับผิดน้อยสุดใน 23 รูปทดสอบ (ผิดรวม 23 ดอก vs yolo+owl2 32, v5 31)   # "yolo" = YOLO ที่ train เอง | "owl2" = OWLv2 zero-shot (ไม่ต้อง train)
                    # "yolo+owl2" = YOLO เป็นหลัก + เติมดอกที่ OWLv2 เจอแต่ YOLO ไม่เจอ (ช้ากว่า ~2 เท่า) | "owl" = OWL-ViT รุ่นเก่า
YOLO_WEIGHTS = "runs/detect/flower_single_yolo11/weights/best.pt"   # YOLOv8 เดิม: "runs/detect/flower_single/weights/best.pt"
OWL_EXTRA_QUERIES = ["flower"]   # OWL ค้นด้วยคำเหล่านี้ + ชื่อสามัญของทุกชนิดใน flowers (อย่าใช้ "a photo of ..." จะได้กล่องครอบทั้งรูป)

SAM3_PROMPT = "flower"    # (sam3) คำที่ให้ SAM 3 หา
SAM3_CONF = 0.6           # (sam3) คะแนนขั้นต่ำ (0.5 นับดอกฟิลเลอร์/ดอกย่อยเกิน, 0.7 เริ่มหลุดดอกจริง)
SAM3_MIN_REL = 0.05
SPECIES_HEAD = "runs/classify/species_bioclip_head_v2.pt"   # ส่วนหัวที่เราเทรนบน BioCLIP 2 (v2: other = แดนดิไลออน + Oxford 102) | "" = ใช้ zero-shot แบบเดิม
GUARD_THR = 1.01  # ด่านกัน: zero-shot มั่นใจ ≥ นี้ว่าเป็นชนิดนอก 4 ชนิด → unknown | 1.01 = ปิด (head v2 ปัดดอกที่ไม่รู้จักได้ 99% เอง) | head v1 ใช้ 0.9
DEMO_SPECIES = {"rose", "tulip", "sunflower", "daisy"}   # ทายจากรายชื่อเต็ม แล้วชนิดที่ไม่อยู่ในนี้ = unknown (ไม่บังคับให้ทุกดอกเป็น 1 ใน 4)       # (sam3) ตัดกล่องที่เล็กกว่า 5% ของกล่องใหญ่สุดในรูป (ยิปโซ/โกลเด้นรอด/ดอกย่อย)

OWL_CONF = 0.10           # คะแนนขั้นต่ำของ OWL
YOLO_CONF = 0.15          # คะแนนขั้นต่ำของ YOLO (0.25 หลุดดอกด้านหลัง/ดอกที่ไม่มีใน dataset)
YOLO_TTA = True          # (YOLO) ทายหลายสเกล + กลับซ้ายขวา แล้วรวมผล เจอดอกครบขึ้น แต่ช้าลง ~2–3 เท่า
DET_REL = 0.25            # (OWL) เก็บกล่องที่คะแนน ≥ DET_REL × คะแนนสูงสุดในรูปนั้น เพราะคะแนน OWLv2 ต่างกันมากในแต่ละรูป
DET_IOU = 0.3
MIN_BOX_PX = 16
MIN_REL_AREA = 0.15      # ตัดกล่องที่เล็กกว่า 15% ของกล่องขนาดกลางในรูป (เศษกลีบ) | 0 = ปิด
DET_MAX_AREA = 0.9
DROP_GROUP_BOXES = True   # ตัดกล่องใหญ่ที่ครอบหลายดอก (ทั้งช่อ) ไม่ให้นับเป็น 1 ดอก
NEST_COVER = 0.7          # กล่องที่อยู่ในอีกกล่อง ≥ 70% = ดอกเดียวกัน (กลีบในดอก, ดอก+ก้าน) → นับ 1
INNER_KEEP = 0.8          # กล่องในได้คะแนน ≥ 80% ของกล่องนอก → เลือกกล่องใน (ตัวดอก ไม่เอาก้าน/ใบ)

# ---------- ตัวจำแนกชนิด (บอกว่าเป็น "ดอกอะไร") ----------
SPECIES_MODEL = "hf-hub:imageomics/bioclip-2"   # รุ่นเดิม: "hf-hub:imageomics/bioclip"
SPECIES_THR = 0.35     # ความมั่นใจต่ำกว่านี้ = unknown
CONTEXT_WEIGHT = 0.5   # ผสมผลของ "ทั้งรูป" เข้ากับแต่ละดอก (0 = ปิด) ใช้เฉพาะรูปชนิดเดียว (ดอกเข็มที่ 0.3 ถูกทายเป็นลีลาวดี)
CONTEXT_MIN = 0.8      # ใช้บริบทเฉพาะเมื่อทั้งรูปเป็นชนิดเดียวชัดๆ (รูปชนิดเดียวได้ ~1.0, ช่อผสมได้ 0.36–0.60)
REF_DIR = "refs"       # รูปตัวอย่าง refs/<ชื่อชนิด>/*.jpg (ไม่มีก็ได้ ดูเซลล์ 5.1)
REF_WEIGHT = 0.6       # น้ำหนักของรูปตัวอย่าง เทียบกับข้อความ (0–1)
SHOW_TOP3 = True       # พิมพ์ 3 อันดับแรกของแต่ละดอก ไว้ดูว่าโมเดลลังเลระหว่างอะไร

# ---------- ภาพจริง / การ์ตูน-เกม / ดอกไม้ปลอม ----------
DRAWN_THR = 0.8               # P(การ์ตูน/เกม/ภาพวาด) เกินนี้ → ไม่นับ (รูปเกมได้ 0.93–1.0, ภาพถ่ายจริงพื้นดำได้ ~0.7)
ARTIFICIAL_THR = 0.8          # P(ดอกไม้ปลอม | เป็นภาพถ่าย) เกินนี้ → โหมดดอกไม้ปลอม (ดอกลวดกำมะหยี่ได้ ~0.82, ดอกจริงได้ถึง ~0.76)
ARTIFICIAL_CLIP = "ViT-L-14"  # CLIP ทั่วไปที่ใช้ช่วยบอกชนิดดอกไม้ปลอม (โหลดเฉพาะตอนเจอ)
ARTIFICIAL_CLIP_WEIGHT = 0.5  # ในโหมดดอกไม้ปลอม เชื่อ CLIP เท่าไร (ที่เหลือเป็น BioCLIP)

# ---------- ด่านกรอง "เป็นดอกไม้ไหม" ----------
IMAGE_FLOWER_THR = 0.30
CROP_FLOWER_THR = 0.5   # แจกัน/ใบไม้/กิ่ง ได้ 0.03–0.35, ดอกจริงได้ ≥ 0.54
UNKNOWN = "unknown"

# ---------- เครื่องเจนรูปแยก (SDXL) ----------
GEN_URL = ""   # ลิงก์ trycloudflare จาก bloom_craft_gen_sdxl.ipynb (รันอีกบัญชี) | "" = เจนด้วย SD 1.5 ในเครื่องนี้

# ---------- ดอกช่อ / ดอกช่อยาว ----------
COUNT_CLUSTER_AS_ONE = True   # True = 1 ช่อ = 1 ดอก | False = นับทีละกล่องดอกย่อย
CLUSTER_SPECIES = {"ixora", "hydrangea", "lilac", "snapdragon", "delphinium", "lavender", "hyacinth",
                   "gladiolus", "stock", "baby's breath", "goldenrod", "statice", "lily of the valley", "bougainvillea"}

## 3) โหลดโมเดล

In [ ]:
import os
import torch, open_clip
from torchvision.ops import nms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# CLIP ทั่วไป: ตัดสินว่า "เป็นดอกไม้ไหม" (ใช้รุ่น quickgelu ให้ตรงกับน้ำหนักของ OpenAI)
gate_model, _, gate_preprocess = open_clip.create_model_and_transforms("ViT-B-32-quickgelu", pretrained="openai")
gate_tok = open_clip.get_tokenizer("ViT-B-32-quickgelu")
gate_model = gate_model.to(device).eval()

# BioCLIP 2: ทายชนิดดอกไม้ (เทรนจากภาพสิ่งมีชีวิตจริงกว่า 200 ล้านรูป ด้วยชื่อวิทยาศาสตร์)
bio_model, _, bio_preprocess = open_clip.create_model_and_transforms(SPECIES_MODEL)
bio_tok = open_clip.get_tokenizer(SPECIES_MODEL)
bio_model = bio_model.to(device).eval()

# Detector
if DETECTOR not in ("sam3", "owl2", "owl", "yolo", "yolo+owl2"):
    raise ValueError("DETECTOR ต้องเป็น 'sam3', 'owl2', 'yolo', 'yolo+owl2' หรือ 'owl'")
if DETECTOR == "sam3":
    try:   # Colab: ล็อกอินด้วย Secret HF_TOKEN ตรงๆ → ถ้าผิดจะบอกสาเหตุชัด แทน 401 ตอนโหลดโมเดล
        from google.colab import userdata
        from huggingface_hub import login, whoami
        try:
            _tok = userdata.get("HF_TOKEN")
        except Exception as e:   # ไม่ได้กด Grant access / ไม่มี Secret → ให้วาง token เองแทนการค้าง
            print("อ่าน Secret HF_TOKEN ไม่ได้:", type(e).__name__)
            from getpass import getpass
            _tok = getpass("วาง Hugging Face token (hf_...) แล้วกด Enter: ")
        login(token=_tok.strip())
        print("ล็อกอิน Hugging Face แล้ว:", whoami()["name"])
    except ImportError:
        pass   # ไม่ใช่ Colab → ใช้ hf auth login ตามปกติ
    from transformers import Sam3Processor, Sam3Model
    sam3_proc = Sam3Processor.from_pretrained("facebook/sam3")
    sam3_model = Sam3Model.from_pretrained("facebook/sam3", dtype=torch.float16 if device == "cuda" else torch.float32,
                                           low_cpu_mem_usage=True).to(device).eval()
    if device == "cuda":   # ตัวใหญ่ 2 ตัวเป็น fp16 (VRAM ~3.4GB) ที่เหลือ fp32 เพราะ fp16 ทำ mask เป็น NaN
        def _f32(x):
            if torch.is_tensor(x) and x.is_floating_point():
                return x.float()
            if isinstance(x, (list, tuple)):
                return type(x)(_f32(t) for t in x)
            if hasattr(x, "keys"):
                for k in list(x.keys()):
                    x[k] = _f32(x[k])
            return x
        sam3_model.float()
        for _m in (sam3_model.vision_encoder.backbone, sam3_model.text_encoder):
            _m.half().register_forward_hook(lambda mod, a, o: _f32(o))
if DETECTOR in ("owl2", "yolo+owl2"):
    from transformers import Owlv2Processor, Owlv2ForObjectDetection
    owl_proc = Owlv2Processor.from_pretrained("google/owlv2-base-patch16-ensemble")
    owl_model = Owlv2ForObjectDetection.from_pretrained("google/owlv2-base-patch16-ensemble").to(device).eval()
elif DETECTOR == "owl":
    from transformers import OwlViTProcessor, OwlViTForObjectDetection
    owl_proc = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
    owl_model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32").to(device).eval()
if DETECTOR in ("yolo", "yolo+owl2"):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    from ultralytics import YOLO
    if not os.path.exists(YOLO_WEIGHTS):
        raise FileNotFoundError(f"ไม่มีไฟล์ {YOLO_WEIGHTS} — train ในเซลล์ 1 (ต้องมี GPU เช่น Colab) แล้ววาง best.pt ไว้ที่นี่ "
                                "หรือตั้ง DETECTOR = \"owl2\" ในเซลล์ 2")
    yolo_det = YOLO(YOLO_WEIGHTS)

# ดาวน์โหลด CLIP ของโหมดดอกไม้ปลอมไว้ก่อน (1.7GB ครั้งแรกครั้งเดียว) ไม่งั้นจะไปดาวน์โหลดกลางเซลล์สรุปตอนเจอดอกไม้ปลอมรูปแรก
# โหลดแค่ไฟล์ลงดิสก์ ไม่โหลดเข้า RAM (โมเดลจะเข้า RAM ตอนเจอดอกไม้ปลอมจริงเท่านั้น)
if ARTIFICIAL_CLIP_WEIGHT > 0:
    open_clip.pretrained.download_pretrained(open_clip.pretrained.get_pretrained_cfg(ARTIFICIAL_CLIP, "openai"))

species_head = None
if SPECIES_HEAD and os.path.exists(SPECIES_HEAD):
    _ck = torch.load(SPECIES_HEAD, map_location=device)
    species_head = torch.nn.Linear(_ck["state_dict"]["weight"].shape[1], len(_ck["classes"])).to(device).eval()
    species_head.load_state_dict(_ck["state_dict"])
    head_classes, head_scale = _ck["classes"], _ck["scale"]
print("species head:", SPECIES_HEAD if species_head is not None else "ไม่มี → zero-shot")
print("Models loaded! detector =", DETECTOR, "| species model =", SPECIES_MODEL)

## 4) ฟังก์ชันช่วย: embedding / detect / crop

In [ ]:
@torch.inference_mode()
def encode_text(model, tok, prompts):
    t = tok(prompts).to(device)
    f = model.encode_text(t)
    return f / f.norm(dim=-1, keepdim=True)


@torch.inference_mode()
def encode_image(model, preprocess, img):
    x = preprocess(img).unsqueeze(0).to(device)
    f = model.encode_image(x)
    return (f / f.norm(dim=-1, keepdim=True))[0]


def _covered(inner, outer):
    """สัดส่วนพื้นที่ของกล่อง inner ที่อยู่ในกล่อง outer"""
    ix0, iy0 = max(inner[0], outer[0]), max(inner[1], outer[1])
    ix1, iy1 = min(inner[2], outer[2]), min(inner[3], outer[3])
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    area = max(1e-6, (inner[2] - inner[0]) * (inner[3] - inner[1]))
    return inter / area


def drop_group_boxes(cand, min_inside=2):
    """ตัดกล่องที่มีกล่องอื่นอยู่ข้างในตั้งแต่ 2 กล่องขึ้นไป (มักเป็นกล่องทั้งช่อ ไม่ใช่ดอกเดียว)"""
    out = []
    for i, (bi, si) in enumerate(cand):
        inside = sum(1 for j, (bj, _) in enumerate(cand) if j != i and _covered(bj, bi) >= NEST_COVER)
        if inside < min_inside:
            out.append((bi, si))
    return out


def drop_nested_boxes(cand):
    """กล่องที่ซ้อนกันเกิน NEST_COVER = ดอกเดียวกัน → เหลือกล่องเดียว
    ปกติเก็บกล่องที่คะแนนสูงกว่า แต่ถ้ากล่องในคะแนนใกล้กัน (≥ INNER_KEEP) เลือกกล่องใน เพราะกล่องนอกมักรวมก้าน/ใบ"""
    out = []
    for bi, si in sorted(cand, key=lambda c: -c[1]):
        j = next((j for j, (bj, _) in enumerate(out)
                  if _covered(bi, bj) >= NEST_COVER or _covered(bj, bi) >= NEST_COVER), None)
        if j is None:
            out.append((bi, si))
        elif _covered(bi, out[j][0]) >= NEST_COVER and si >= INNER_KEEP * out[j][1]:
            out[j] = (bi, si)
    return out


def drop_tiny_boxes(cand):
    """ตัดกล่องที่เล็กผิดปกติเทียบกับกล่องขนาดกลางในรูป (มักเป็นเศษกลีบของดอกข้างๆ)
    ต้องมี ≥ 3 กล่อง ไม่งั้นค่ากลางไม่มีความหมาย"""
    rel = MIN_REL_AREA
    if rel <= 0 or len(cand) < 3:
        return cand
    areas = sorted((b[2] - b[0]) * (b[3] - b[1]) for b, _ in cand)
    med = areas[len(areas) // 2]
    return [(b, s) for b, s in cand if (b[2] - b[0]) * (b[3] - b[1]) >= rel * med]


def owl_queries():
    """คำค้นของ OWL: "flower" + ชื่อสามัญของชนิดที่กำลังทาย (names มาจากเซลล์ 5)"""
    return list(dict.fromkeys(OWL_EXTRA_QUERIES + [flowers[n][1] for n in names]))


def _owl_cand(img):
    """OWLv2 / OWL-ViT → list ของ ((x0, y0, x1, y1), score)"""
    W, H = img.size
    inputs = owl_proc(text=[owl_queries()], images=img, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = owl_model(**inputs)

    if DETECTOR != "owl":
        # OWLv2 เติมขอบให้รูปเป็นสี่เหลี่ยมจัตุรัสก่อนเข้าโมเดล → ต้องแปลงพิกัดด้วยด้านที่ยาวที่สุด
        S = max(W, H)
        sizes = torch.tensor([[S, S]], device=device)
    else:
        sizes = torch.tensor([[H, W]], device=device)

    r = owl_proc.image_processor.post_process_object_detection(
        outputs=outputs, threshold=OWL_CONF, target_sizes=sizes)[0]
    if len(r["boxes"]) == 0:
        return []

    rel = r["scores"] >= DET_REL * r["scores"].max()   # เกณฑ์ปรับตามรูป
    boxes, scores = r["boxes"][rel].clone(), r["scores"][rel]
    boxes[:, 0::2] = boxes[:, 0::2].clamp(0, W)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(0, H)
    keep = nms(boxes, scores, DET_IOU).tolist()   # รวมกล่องซ้ำจากหลาย query
    return [(boxes[i].tolist(), scores[i].item()) for i in keep]


def _sam3_cand(img):
    """SAM 3 → list ของ ((x0, y0, x1, y1), score) หลังตัดคะแนนต่ำ + กล่องจิ๋วเทียบกล่องใหญ่สุด"""
    inputs = sam3_proc(images=img, text=SAM3_PROMPT, return_tensors="pt").to(device)
    inputs["pixel_values"] = inputs["pixel_values"].to(next(sam3_model.vision_encoder.backbone.parameters()).dtype)
    with torch.inference_mode():
        outputs = sam3_model(**inputs)
    r = sam3_proc.post_process_instance_segmentation(outputs, threshold=SAM3_CONF, mask_threshold=0.5,
                                                     target_sizes=inputs["original_sizes"].tolist())[0]
    W, H = img.size
    cand = [([max(0, b[0]), max(0, b[1]), min(W, b[2]), min(H, b[3])], s, m) for b, s, m in
            zip(r["boxes"].float().cpu().tolist(), r["scores"].float().cpu().tolist(), r["masks"].cpu().numpy())]
    if not cand:
        return []
    big = max((b[2] - b[0]) * (b[3] - b[1]) for b, _, _ in cand)
    cand = [c for c in cand if (c[0][2] - c[0][0]) * (c[0][3] - c[0][1]) >= SAM3_MIN_REL * big]
    _sam3_masks.clear()   # mask ของรูปปัจจุบัน (ตัดเฉพาะในกล่อง) ไว้ให้ flower_color / cutout ใช้
    for b, _, m in cand:
        x0, y0, x1, y1 = _bkey(b)
        _sam3_masks[(x0, y0, x1, y1)] = m[y0:y1, x0:x1].astype(bool)
    return [(b, s) for b, s, _ in cand]


_sam3_masks = {}


def _bkey(box):
    return tuple(int(round(v)) for v in box)


def _yolo_cand(img):
    """YOLO → list ของ ((x0, y0, x1, y1), score)"""
    results = yolo_det(img, conf=YOLO_CONF, iou=DET_IOU, augment=YOLO_TTA, verbose=False)[0]
    return [(box.xyxy[0].tolist(), box.conf[0].item()) for box in results.boxes]


def _iou(a, b):
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    inter = _covered(a, b) * area_a
    return inter / max(1e-6, area_a + area_b - inter)


def _much_smaller(inner, outer):
    return (inner[2] - inner[0]) < 0.5 * (outer[2] - outer[0]) and (inner[3] - inner[1]) < 0.5 * (outer[3] - outer[1])


def merge_yolo_owl(yolo, owl):
    """เก็บกล่อง YOLO + กล่อง OWL ที่ YOLO ไม่เจอ (คะแนน 2 โมเดลเทียบกันตรงๆ ไม่ได้ จึงไม่ใช้ NMS รวม)
    - OWL ซ้ำกับ YOLO (IoU ≥ DET_IOU หรืออยู่ในกล่อง YOLO) → ทิ้ง OWL
    - OWL ครอบกล่อง YOLO ≥ 2 กล่อง = กล่องทั้งช่อ → ทิ้ง OWL
    - OWL ครอบกล่อง YOLO ที่เล็กกว่าครึ่งทั้งกว้างและสูง 1 กล่อง = ดอกใหญ่ที่ YOLO เจอแค่ของเล็กๆ ข้างใน → ใช้กล่อง OWL แทน
      (เฉพาะกล่อง OWL ที่คะแนน ≥ 60% ของคะแนนสูงสุดในรูป)"""
    out = list(yolo)
    owl_max = max((s for _, s in owl), default=0)
    for o in owl:
        if any(_iou(o[0], y[0]) >= DET_IOU or _covered(o[0], y[0]) >= NEST_COVER for y in yolo):
            continue
        inside = [y for y in yolo if _covered(y[0], o[0]) >= NEST_COVER]
        if len(inside) >= 2:
            continue
        if inside and not _much_smaller(inside[0][0], o[0]):   # ดอก+ก้าน (กว้างเท่ากัน) = ดอกเดียวกัน → เก็บกล่อง YOLO
            continue
        if inside and o[1] < 0.6 * owl_max:   # OWL ไม่มั่นใจ (เช่น กล่องแจกัน+ดอกบน ได้ 0.4× ส่วนทานตะวันได้ 1.0×) → เก็บกล่อง YOLO
            continue
        out = [c for c in out if c not in inside] + [o]
    return out


def detect_flowers(img):
    """ตรวจจับดอกไม้ รองรับ OWLv2 / YOLO / YOLO+OWLv2 / OWL-ViT
    → (กล่องดอก, กล่องที่ครอบหลายกล่อง) แต่ละอันเป็น list ของ ((x0, y0, x1, y1), score)"""
    W, H = img.size
    if DETECTOR == "sam3":
        cand = _sam3_cand(img)
    elif DETECTOR in ("owl", "owl2"):
        cand = _owl_cand(img)
    elif DETECTOR == "yolo":
        cand = _yolo_cand(img)
    elif DETECTOR == "yolo+owl2":
        cand = merge_yolo_owl(_yolo_cand(img), _owl_cand(img))
    else:
        raise ValueError("DETECTOR ต้องเป็น 'owl2', 'yolo', 'yolo+owl2' หรือ 'owl'")

    # กรองกล่องที่เล็กหรือใหญ่เกินไป
    out = []
    for (x0, y0, x1, y1), score in cand:
        w, h = x1 - x0, y1 - y0
        if w < MIN_BOX_PX or h < MIN_BOX_PX:
            continue
        if w * h > DET_MAX_AREA * W * H:
            continue
        out.append(((x0, y0, x1, y1), score))

    if DETECTOR == "sam3":
        return out, []
    out = drop_tiny_boxes(out)   # ก่อน drop_group_boxes: เศษกลีบในดอกริมรูปทำให้ดอกจริงถูกมองเป็นกล่องทั้งช่อ
    groups = []
    if DROP_GROUP_BOXES:
        kept = drop_group_boxes(out)
        groups = [c for c in out if c not in kept]   # เก็บไว้ให้ merge_clusters (ดอกช่อ เช่น ดอกเข็ม คือกล่องแบบนี้)
        out = kept
    return drop_nested_boxes(out), groups


def crop_with_padding(img, box, pad=0.1):
    x0, y0, x1, y1 = box
    W, H = img.size
    px, py = (x1 - x0) * pad, (y1 - y0) * pad
    return img.crop((max(0, int(x0 - px)), max(0, int(y0 - py)),
                     min(W, int(x1 + px)), min(H, int(y1 + py))))

## 5) รายชื่อดอกไม้ + prompt
`flowers` = `"ชื่อที่แสดง": ("ชื่อวิทยาศาสตร์", "ชื่อสามัญ")`

- BioCLIP เรียนรู้จาก **ชื่อวิทยาศาสตร์** เป็นหลัก ถ้าเพิ่มชนิดใหม่ ให้ใส่ชื่อวิทยาศาสตร์ด้วยเสมอ
- **ลบชนิดที่ไม่มีทางเจอออก** ยิ่งตัวเลือกน้อย ยิ่งทายถูก (เช่น ถ้างานนี้มีแค่ดอกไม้จัดช่อ ไม่ต้องมี higanbana)

In [ ]:
import os, glob
from PIL import Image

# ---------- ด่าน "เป็นดอกไม้ไหม" ----------
flower_prompts = [
    "a photo of a flower",
    "a close-up photo of a blooming flower",
    "a photo of a bouquet of flowers",
    "a photo of flowers in a garden",
    "a photo of artificial flowers",   # ให้ดอกไม้ปลอม (พลาสติก/ผ้า) ผ่านด่านนี้
]
other_prompts = [
    "a cartoon drawing", "a 2D game illustration", "a video game character", "an anime illustration",
    "a photo of an animal", "a photo of a person", "a photo of a face", "a photo of a hand",
    "a photo of food", "a photo of fruit", "a photo of a building", "a photo of a car",
    "a photo of a landscape", "a photo of green leaves without flowers", "a photo of a tree",
    "a photo of an everyday object", "a screenshot", "a photo of text or a document",
    "a photo of a ribbon bow", "a photo of gift wrapping paper",   # YOLO (เทรนจากช่อดอกไม้) ชอบจับริบบิ้น/กระดาษห่อ
]
gate_prompts = flower_prompts + other_prompts
n_flower = len(flower_prompts)
gate_text = encode_text(gate_model, gate_tok, gate_prompts)
gate_scale = gate_model.logit_scale.exp().item()

# ---------- ด่าน 2: ภาพถ่ายดอกจริง / ภาพถ่ายดอกไม้ปลอม / การ์ตูน-เกม ----------
real_prompts = ["a photo of a real living flower", "a photo of fresh natural flowers"]
fake_prompts = ["a photo of artificial plastic flowers", "a photo of fake silk flowers",
                "a photo of fabric flowers", "a photo of handmade flowers made of pipe cleaners"]
drawn_prompts = ["a cartoon drawing of flowers", "flowers in a video game screenshot",
                 "a 3D rendered flower from a video game", "an anime illustration of flowers",
                 "a flower clipart icon", "pixel art flowers", "a vector illustration of flowers"]
kind_text = encode_text(gate_model, gate_tok, real_prompts + fake_prompts + drawn_prompts)
_nr, _nf = len(real_prompts), len(fake_prompts)


def photo_kind(img):
    """คืน (P(การ์ตูน/เกม), P(ดอกไม้ปลอม | เป็นภาพถ่าย))"""
    g = encode_image(gate_model, gate_preprocess, img)
    p = (gate_scale * (g @ kind_text.T)).softmax(dim=-1)
    p_real, p_fake, p_drawn = p[:_nr].sum().item(), p[_nr:_nr + _nf].sum().item(), p[_nr + _nf:].sum().item()
    return p_drawn, p_fake / max(1e-6, p_real + p_fake)

# ---------- ชนิดดอกไม้: "ชื่อที่แสดง": ("ชื่อวิทยาศาสตร์", "ชื่อสามัญ") ----------
flowers = {
    # --- ดอกไม้จัดช่อยอดนิยม ---
    "rose": ("Rosa", "rose"),
    "tulip": ("Tulipa gesneriana", "tulip"),
    "lily": ("Lilium", "lily"),
    "orchid": ("Orchidaceae", "orchid"),
    "peony": ("Paeonia lactiflora", "peony"),
    "iris": ("Iris", "iris"),
    "chrysanthemum": ("Chrysanthemum morifolium", "chrysanthemum"),
    "lisianthus": ("Eustoma grandiflorum", "lisianthus"),
    "sunflower": ("Helianthus annuus", "sunflower"),
    "carnation": ("Dianthus caryophyllus", "carnation"),
    "baby's breath": ("Gypsophila paniculata", "baby's breath"),
    "gerbera": ("Gerbera jamesonii", "gerbera daisy"),
    "hydrangea": ("Hydrangea macrophylla", "hydrangea"),
    "daisy": ("Leucanthemum", "daisy"),
    "dahlia": ("Dahlia", "dahlia"),
    "ranunculus": ("Ranunculus asiaticus", "Persian buttercup"),
    "anemone": ("Anemone coronaria", "poppy anemone"),
    "freesia": ("Freesia", "freesia"),
    "gladiolus": ("Gladiolus", "gladiolus"),
    "alstroemeria": ("Alstroemeria", "Peruvian lily"),
    "calla lily": ("Zantedeschia", "calla lily"),
    "daffodil": ("Narcissus", "daffodil"),
    "hyacinth": ("Hyacinthus orientalis", "hyacinth"),
    "poppy": ("Papaver", "poppy"),
    "sweet pea": ("Lathyrus odoratus", "sweet pea"),
    "snapdragon": ("Antirrhinum majus", "snapdragon"),
    "delphinium": ("Delphinium", "larkspur"),
    "lilac": ("Syringa vulgaris", "lilac"),
    "lavender": ("Lavandula angustifolia", "lavender"),
    "stock": ("Matthiola incana", "stock"),
    "statice": ("Limonium sinuatum", "statice"),
    "wax flower": ("Chamelaucium uncinatum", "Geraldton wax flower"),
    "anthurium": ("Anthurium andraeanum", "flamingo flower"),
    "protea": ("Protea cynaroides", "king protea"),
    "aster": ("Aster", "aster"),
    "zinnia": ("Zinnia elegans", "zinnia"),
    "cosmos": ("Cosmos bipinnatus", "garden cosmos"),
    "lily of the valley": ("Convallaria majalis", "lily of the valley"),
    "forget-me-not": ("Myosotis", "forget-me-not"),
    # --- ดอกไม้ที่เจอบ่อยในไทย/เอเชีย ---
    "plumeria": ("Plumeria rubra", "frangipani"),          # ลีลาวดี
    "lotus": ("Nelumbo nucifera", "sacred lotus"),         # บัว
    "jasmine": ("Jasminum sambac", "Arabian jasmine"),     # มะลิ
    "marigold": ("Tagetes erecta", "African marigold"),    # ดาวเรือง
    "hibiscus": ("Hibiscus rosa-sinensis", "hibiscus"),    # ชบา
    "bougainvillea": ("Bougainvillea", "bougainvillea"),   # เฟื่องฟ้า
    "ixora": ("Ixora coccinea", "jungle geranium"),        # เข็ม
    "gardenia": ("Gardenia jasminoides", "gardenia"),
    "camellia": ("Camellia japonica", "camellia"),
    "magnolia": ("Magnolia", "magnolia"),
    "cherry blossom": ("Prunus serrulata", "Japanese cherry"),
    "higanbana": ("Lycoris radiata", "red spider lily"),
    # --- ดอกเติมช่อ / ช่อผสม ---
    "goldenrod": ("Solidago", "goldenrod"),
    "scabiosa": ("Scabiosa atropurpurea", "pincushion flower"),
    "clematis": ("Clematis", "clematis"),
    "craspedia": ("Craspedia globosa", "billy buttons"),
}

# prompt แบบที่ BioCLIP ใช้ตอนเทรน: ชื่อวิทยาศาสตร์ + ชื่อสามัญ
templates = [
    "a photo of {sci}.",
    "a photo of {sci} with common name {common}.",
    "a photo of {common} flowers.",
    "a close-up photo of a {common} flower.",
]
IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.webp", "*.JPG", "*.JPEG", "*.PNG", "*.WEBP")


def _text_emb(name):
    sci, common = flowers[name]
    f = encode_text(bio_model, bio_tok, [t.format(sci=sci, common=common) for t in templates]).mean(dim=0)
    return f / f.norm()


def _ref_paths(name):
    paths = []
    for ext in IMG_EXTS:
        paths += glob.glob(os.path.join(REF_DIR, name, ext))
    return sorted(set(paths))


def _ref_emb(paths):
    fs = []
    for p in paths:
        try:
            fs.append(encode_image(bio_model, bio_preprocess, Image.open(p).convert("RGB")))
        except Exception:
            print("⚠️ อ่านรูปตัวอย่างไม่ได้ ข้าม:", p)
    if not fs:
        return None
    f = torch.stack(fs).mean(dim=0)
    return f / f.norm()


def build_class_embs():
    """สร้าง embedding ของแต่ละชนิด จากข้อความ (+ รูปตัวอย่าง ถ้ามีโฟลเดอร์ refs/)"""
    global names, class_embs
    refs = {n: _ref_paths(n) for n in flowers}
    with_refs = [n for n in flowers if refs[n]]

    if os.path.isdir(REF_DIR):
        extra = [d for d in sorted(os.listdir(REF_DIR))
                 if os.path.isdir(os.path.join(REF_DIR, d)) and d not in flowers]
        if extra:
            print("⚠️ ชื่อโฟลเดอร์ไม่ตรงกับชื่อใน flowers (แก้ชื่อโฟลเดอร์ หรือเพิ่มชนิดใน flowers):", extra)

    if with_refs:
        # มีรูปตัวอย่าง → ทายเฉพาะชนิดที่มีรูปตัวอย่าง
        # (ถ้าปนกับชนิดที่มีแต่ข้อความ ชนิดที่มีรูปจะได้คะแนนสูงกว่าเสมอ ผลจะเอียง)
        names = with_refs
        embs = []
        for n in names:
            t, r = _text_emb(n), _ref_emb(refs[n])
            e = t if r is None else (1 - REF_WEIGHT) * t + REF_WEIGHT * r
            embs.append(e / e.norm())
        print(f"ใช้ข้อความ + รูปตัวอย่าง: {len(names)} ชนิด → "
              + ", ".join(f"{n} ({len(refs[n])} รูป)" for n in names))
    else:
        names = list(flowers)
        embs = [_text_emb(n) for n in names]
        print(f"ใช้ข้อความอย่างเดียว: {len(names)} ชนิด (ยังไม่มีรูปตัวอย่างใน {REF_DIR}/)")
    class_embs = torch.stack(embs)


build_class_embs()
bio_scale = bio_model.logit_scale.exp().item()
print("prompt ด่านกรอง:", len(gate_prompts))

## 5.1) (ไม่บังคับ แต่แม่นขึ้นมากที่สุด) อัปโหลดรูปตัวอย่าง
ทำไฟล์ `.zip` ที่ข้างในเป็นโฟลเดอร์ตามชื่อชนิด (ต้องตรงกับ key ใน `flowers`) เช่น

```
refs.zip
 └─ refs/
     ├─ tulip/      ← รูปทิวลิป 10–30 รูป (หลายสี หลายมุม ทั้งดอกตูมและบาน)
     ├─ rose/
     └─ calla lily/
```
- ใช้รูปที่ **เห็นดอกเดี่ยวชัดๆ** (crop มาแล้วยิ่งดี) และ **อย่าใช้รูปเดียวกับที่จะทดสอบ**
- เมื่อมีรูปตัวอย่าง ระบบจะทาย **เฉพาะชนิดที่มีโฟลเดอร์** เท่านั้น
- รันเซลล์นี้ซ้ำได้เพื่อเพิ่มรูป ระบบจะสร้าง embedding ใหม่ให้อัตโนมัติ

In [ ]:
import io, zipfile
try:
    from google.colab import files
except ImportError:   # VS Code / เครื่องตัวเอง: วางรูปใน refs/<ชนิด>/ ได้เลย ไม่ต้อง zip
    files = None


def upload_refs():
    if files is None:
        print(f"(VS Code) อ่านรูปตัวอย่างจากโฟลเดอร์ {REF_DIR}/<ชนิด>/ โดยตรง")
        build_class_embs()
        return
    up = files.upload()
    added = 0
    for fname, data in up.items():
        if not fname.lower().endswith(".zip"):
            print("⚠️ ข้าม (ไม่ใช่ .zip):", fname)
            continue
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            members = [m for m in z.namelist()
                       if not m.endswith("/") and not m.startswith("__MACOSX")
                       and not os.path.basename(m).startswith(".")]
            tops = {m.split("/")[0] for m in members}
            strip = len(tops) == 1 and next(iter(tops)) not in flowers   # zip มีโฟลเดอร์ครอบอีกชั้น เช่น refs/
            for m in members:
                parts = m.split("/")[1:] if strip else m.split("/")
                if len(parts) != 2 or ".." in parts:   # ต้องเป็น <ชนิด>/<รูป>
                    continue
                os.makedirs(os.path.join(REF_DIR, parts[0]), exist_ok=True)
                with open(os.path.join(REF_DIR, *parts), "wb") as f:
                    f.write(z.read(m))
                added += 1
    print(f"✅ เพิ่มรูปตัวอย่าง {added} รูป")
    build_class_embs()


if files is None:
    upload_refs()
else:   # Colab: ไม่เปิดช่องอัปโหลดเอง ไม่งั้น Run all ค้างรอ
    print("ข้ามได้ ถ้าจะอัปโหลดรูปตัวอย่าง (.zip) ให้พิมพ์ upload_refs() ในเซลล์ใหม่")


## 6) วิเคราะห์รูปเดียว: `analyze_image`

In [ ]:
def flower_prob(img):
    """P(เป็นดอกไม้) จาก CLIP ทั่วไป + สิ่งที่ใกล้เคียงที่สุดในกลุ่ม 'ไม่ใช่ดอกไม้'"""
    g = encode_image(gate_model, gate_preprocess, img)
    p = (gate_scale * (g @ gate_text.T)).softmax(dim=-1)
    j = p[n_flower:].argmax().item()
    return p[:n_flower].sum().item(), (other_prompts[j], p[n_flower + j].item())


_art = {}


def _art_clip():
    """โหลด CLIP ViT-L-14 ครั้งแรกที่เจอดอกไม้ปลอม และสร้าง embedding ใหม่ถ้ารายชื่อชนิดเปลี่ยน"""
    if "model" not in _art:
        m, _, pre = open_clip.create_model_and_transforms(ARTIFICIAL_CLIP, pretrained="openai", force_quick_gelu=True)
        _art.update(model=m.to(device).eval(), pre=pre, tok=open_clip.get_tokenizer(ARTIFICIAL_CLIP),
                    scale=m.logit_scale.exp().item())
    if _art.get("names") != names:
        tmpl = ["a photo of an artificial {} flower made of plastic.", "a photo of a fake silk {} flower.",
                "a photo of a {} flower made of pipe cleaners.", "a photo of a {} flower."]
        embs = []
        for n in names:
            f = encode_text(_art["model"], _art["tok"], [t.format(n) for t in tmpl]).mean(dim=0)
            embs.append(f / f.norm())
        _art.update(embs=torch.stack(embs), names=list(names))
    return _art


def species_logits(img, artificial=False):
    b = encode_image(bio_model, bio_preprocess, img)
    logits = bio_scale * (class_embs @ b)
    if artificial and ARTIFICIAL_CLIP_WEIGHT > 0:
        # BioCLIP เรียนจากดอกจริงเท่านั้น → ดอกไม้ปลอมให้ CLIP ทั่วไปช่วย
        a = _art_clip()
        g = encode_image(a["model"], a["pre"], img)
        logits = (1 - ARTIFICIAL_CLIP_WEIGHT) * logits + ARTIFICIAL_CLIP_WEIGHT * (a["scale"] * (a["embs"] @ g))
    return logits


def classify_species(img, context=None, artificial=False):
    """คืน (ชื่อชนิดหรือ UNKNOWN, ความมั่นใจ, top3)
    context = logits ของทั้งรูป เอามาผสม เพื่อให้ดอกในรูปเดียวกันไม่กระจัดกระจายไปคนละชนิด"""
    logits = species_logits(img, artificial)
    if context is not None and CONTEXT_WEIGHT > 0:
        logits = (1 - CONTEXT_WEIGHT) * logits + CONTEXT_WEIGHT * context
    probs = logits.softmax(dim=-1)
    order = probs.argsort(descending=True).tolist()
    top3 = [(names[i], round(probs[i].item(), 4)) for i in order[:3]]
    best = top3[0]
    if species_head is not None and not artificial:   # ดอกไม้ปลอม: head เทรนจากดอกจริง → ใช้ zero-shot + CLIP แบบเดิม
        if best[0] not in DEMO_SPECIES and best[1] >= GUARD_THR:
            return UNKNOWN, best[1], top3
        with torch.inference_mode():
            hp = species_head(encode_image(bio_model, bio_preprocess, img).float() * head_scale).softmax(-1)
        o = hp.argsort(descending=True).tolist()
        top3 = [(head_classes[i], round(hp[i].item(), 4)) for i in o[:3]]
        return (UNKNOWN if top3[0][0] == "other" else top3[0][0]), top3[0][1], top3
    return (best[0] if best[1] >= SPECIES_THR else UNKNOWN), best[1], top3


import colorsys
import numpy as np


def flower_color(img, box):
    """สีหลักของดอก → ((r, g, b), ชื่อสี)
    ค่าเฉลี่ยทั้งกล่องจะปนสีใบไม้/พื้นหลัง และเฉลี่ยสีต่างกันจนได้สีที่ไม่มีอยู่จริง จึง
    ใช้กลางกล่อง 70% → ตัดใบไม้และพิกเซลมืด → เลือกโทนสีที่มีมากที่สุด → ค่ากลางของครึ่งที่สว่างกว่า"""
    x0, y0, x1, y1 = box
    m = _sam3_masks.get(_bkey(box)) if DETECTOR == "sam3" else None
    if m is not None and m.sum() >= 50:   # SAM 3: ใช้เฉพาะพิกเซลในรูปทรงดอก ไม่ปนพื้นหลัง/ดอกข้างๆ
        crop = img.crop(_bkey(box))
        rgb = np.asarray(crop, dtype=np.float32)[m]
        h, s, v = (np.asarray(crop.convert("HSV"), dtype=np.float32)[m] / 255).T
    else:
        dx, dy = (x1 - x0) * 0.15, (y1 - y0) * 0.15
        crop = img.crop((int(x0 + dx), int(y0 + dy), int(x1 - dx), int(y1 - dy)))
        rgb = np.asarray(crop, dtype=np.float32).reshape(-1, 3)
        h, s, v = (np.asarray(crop.convert("HSV"), dtype=np.float32).reshape(-1, 3) / 255).T   # PIL: hue 0–255 = 0–360°
    keep = (v > 0.2) & ~((h > 0.19) & (h < 0.56) & (s > 0.15))   # ไม่เอาใบไม้ (เขียว–เขียวอมฟ้า 70°–200°) และพิกเซลมืด
    if keep.sum() < 0.05 * len(rgb):                            # เหลือน้อยเกินไป (ดอกสีเขียว/มืด) → ใช้ทั้งหมด
        keep[:] = True
    pale = keep & (s < 0.15)
    if pale.sum() >= 0.5 * keep.sum():                          # ส่วนใหญ่ไม่มีสี → ดอกขาว/เทา
        pick = pale
    else:
        bins = (h * 12).astype(int) % 12                        # 12 โทน โทนละ 30°
        sat = keep & (s >= 0.15)
        pick = sat & (bins == np.bincount(bins[sat], minlength=12).argmax())
    pick &= v >= np.median(v[pick])                             # ครึ่งที่สว่างกว่า: คนดูสีจากกลีบที่โดนแสง ไม่ใช่เงา
    r, g, b = (int(x) for x in np.median(rgb[pick], axis=0))
    return (r, g, b), color_name(r, g, b)


def color_name(r, g, b):
    h, s, v = colorsys.rgb_to_hsv(r / 255, g / 255, b / 255)
    h *= 360
    if v < 0.2:
        return "black"
    if s < 0.15:
        return "white" if v > 0.45 else "grey"
    if s < 0.3 and v > 0.85 and 30 <= h < 70:   # ครีม/งาช้าง → white
        return "white"
    if h < 15 or h >= 345:
        return "pink" if s < 0.55 and v > 0.7 else "red"
    if h < 33:
        return "orange"
    if h < 70:
        return "yellow"
    if h < 170:
        return "green"
    if h < 255:
        return "blue"
    if h < 290 or v < 0.75:                                     # ม่วงอมชมพูเข้ม (ทิวลิปม่วง ~0.72) = purple, สว่าง (ชมพู ≥ 0.9) = pink
        return "purple"
    return "pink"


def merge_clusters(img, items, groups, ctx, artificial):
    """ดอกช่อ: กล่องที่ครอบหลายกล่องและทายได้เป็นชนิดใน CLUSTER_SPECIES → รวมดอกย่อยข้างในเป็น 1 ดอก
    ต้องมีดอกย่อยข้างใน ≥ ครึ่งเป็นชนิดเดียวกัน กันช่อผสมทั้งช่อถูกรวมเป็นดอกเดียว"""
    for box, score in sorted(groups, key=lambda g: (g[0][2] - g[0][0]) * (g[0][3] - g[0][1])):   # ช่อเล็กก่อน
        inner = [it for it in items if _covered(it["box"], box) >= NEST_COVER]
        if len(inner) < 2 or sum(1 for it in inner if it.get("cluster")) >= 2:   # ครอบหลายช่อ = ไม่ใช่ช่อเดียว
            continue
        label, conf, top3 = classify_species(crop_with_padding(img, box), ctx, artificial)
        if label not in CLUSTER_SPECIES or sum(it["label"] == label for it in inner) * 2 < len(inner):
            continue
        # ดอกย่อยชนิดเดียวกันที่ล้นขอบกล่องช่อออกไป (กล่องช่อมักสั้นกว่าช่อจริง) ก็เป็นช่อเดียวกัน
        inner += [it for it in items if it not in inner and it["label"] == label and _covered(it["box"], box) >= 0.25]
        items = [it for it in items if it not in inner]
        items.append({"box": [round(v) for v in box], "label": label, "conf": conf, "top3": top3,
                      "det_score": round(score, 3), "cluster": True})
        items[-1]["color"], items[-1]["color_name"] = flower_color(img, box)
    return items


def analyze_image(img):
    p_img, looks_like = flower_prob(img)
    if p_img < IMAGE_FLOWER_THR:
        return {"is_flower": False, "p_flower": p_img, "looks_like": looks_like,
                "items": [], "dropped": 0, "fallback": False, "reason": "not_flower"}

    p_drawn, p_fake = photo_kind(img)
    if p_drawn >= DRAWN_THR:                 # ดอกไม้การ์ตูน / ในเกม / ภาพวาด → ไม่นับ
        return {"is_flower": False, "p_flower": p_img, "looks_like": ("การ์ตูน/เกม/ภาพวาด", p_drawn),
                "items": [], "dropped": 0, "fallback": False, "reason": "drawn"}
    artificial = p_fake >= ARTIFICIAL_THR

    ctx = species_logits(img, artificial) if CONTEXT_WEIGHT > 0 else None   # บริบทของทั้งรูป
    if ctx is not None and ctx.softmax(dim=-1).max() < CONTEXT_MIN:       # ช่อผสมหลายชนิด → ไม่ใช้บริบท
        ctx = None
    items, dropped = [], 0
    boxes, groups = detect_flowers(img)
    for box, det_score in boxes:
        crop = crop_with_padding(img, box)
        p_crop, _ = flower_prob(crop)
        if p_crop < CROP_FLOWER_THR:        # ไม่ใช่ดอกไม้จริง (ใบไม้ มือ ฯลฯ) → ตัดทิ้ง
            dropped += 1
            continue
        label, conf, top3 = classify_species(crop, ctx, artificial)
        rgb, cname = flower_color(img, box)
        items.append({"box": [round(v) for v in box], "label": label, "conf": conf,
                      "top3": top3, "det_score": round(det_score, 3), "color": rgb, "color_name": cname,
                      "mask": _sam3_masks.get(_bkey(box)) if DETECTOR == "sam3" else None})
    if COUNT_CLUSTER_AS_ONE:
        items = merge_clusters(img, items, groups, ctx, artificial)

    fallback = False
    if not items:                           # detector ไม่เจอกล่อง → ประเมินชนิดจากทั้งรูป
        label, conf, top3 = classify_species(img, artificial=artificial)
        if label != UNKNOWN:
            W, H = img.size
            rgb, cname = flower_color(img, (0, 0, W, H))
            items.append({"box": [0, 0, W, H], "label": label, "conf": conf, "top3": top3, "det_score": None,
                          "color": rgb, "color_name": cname})
            fallback = True
        else:
            return {"is_flower": False, "p_flower": p_img, "looks_like": (top3[0][0], top3[0][1]),
                    "items": [], "dropped": dropped, "fallback": False, "reason": "not_flower"}

    for it in items:
        if DEMO_SPECIES and it["label"] not in DEMO_SPECIES:
            it["label"] = UNKNOWN
    return {"is_flower": True, "p_flower": p_img, "looks_like": None, "items": items,
            "dropped": dropped, "fallback": fallback, "artificial": artificial, "p_fake": p_fake}

## 7) ที่เก็บรูป: เพิ่มรูปได้หลายรอบก่อนกดตรวจ
**รันเซลล์นี้ซ้ำเพื่อเพิ่มรูปทีละชุด** (เลือกหลายไฟล์พร้อมกันได้) รูปจะสะสมไว้ในรายการ ยังไม่ตรวจจนกว่าจะรันเซลล์ที่ 8
ถ้าอยากล้างรายการแล้วเริ่มใหม่ ให้รัน `clear_images()`

In [ ]:
import io, os, glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
try:
    from google.colab import files
except ImportError:   # VS Code / เครื่องตัวเอง: อ่านรูปจากโฟลเดอร์แทนการอัปโหลด
    files = None

IMAGE_DIR = "demo_รูป"   # (VS Code) โฟลเดอร์ของรูปที่จะตรวจ
LOCAL_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".avif", ".bmp")

image_bank = []   # list ของ (ชื่อไฟล์, รูป PIL)


def clear_images():
    image_bank.clear()
    print("ล้างรายการรูปแล้ว")


def show_bank(cols=6):
    n = len(image_bank)
    if n == 0:
        print("ยังไม่มีรูป")
        return

    # คำนวณจำนวนแถวและคอลัมน์ให้ยืดหยุ่น
    actual_cols = min(n, cols)
    rows = (n + actual_cols - 1) // actual_cols if actual_cols > 0 else 1

    fig, axes = plt.subplots(rows, actual_cols, figsize=(2.2 * actual_cols, 2.2 * rows))

    # จัดการแกนแสดงผล (axes) ให้เป็น Flat list เสมอไม่ว่าจะอัปโหลดกี่รูป
    if n == 1:
        axes = [axes]
    elif hasattr(axes, 'flatten'):
        axes = axes.flatten()
    else:
        axes = [axes]

    for ax in axes:
        ax.axis("off")

    for ax, (name, im) in zip(axes, image_bank):
        ax.imshow(im)
        ax.set_title(name[:16], fontsize=7)
    plt.tight_layout()
    plt.show()


def add_images(folder=IMAGE_DIR):
    existing = {n for n, _ in image_bank}
    if files is not None and not glob.glob(os.path.join(folder, "*")):
        up = files.upload()
    else:   # รันซ้ำได้ จะเพิ่มเฉพาะไฟล์ใหม่ในโฟลเดอร์
        up = {os.path.basename(f): Path(f).read_bytes()
              for f in sorted(glob.glob(os.path.join(folder, "*")))
              if f.lower().endswith(LOCAL_EXTS) and os.path.basename(f) not in existing}
    for fname, data in up.items():
        try:
            im = Image.open(io.BytesIO(data)).convert("RGB")
        except Exception:
            print("⚠️ อ่านไฟล์ไม่ได้ ข้าม:", fname)
            continue
        name, k = fname, 2
        while name in existing:                  # ชื่อซ้ำ → ต่อท้ายเลข
            name = f"{fname.rsplit('.', 1)[0]}_{k}"
            k += 1
        existing.add(name)
        image_bank.append((name, im))
    print(f"✅ ตอนนี้มีรูปในรายการ {len(image_bank)} รูป")
    show_bank()


add_images()

## 8) กดตรวจทั้งหมด + สรุปรวม

In [ ]:
from collections import Counter, defaultdict
from IPython.display import display, HTML
import matplotlib.patches as patches


def summarize(results):
    """results = list ของ (ชื่อรูป, ผล analyze_image) → สรุปรวมทุกรูป (ฟังก์ชันล้วน ไม่แตะโมเดล)"""
    total, in_images, colors = Counter(), defaultdict(set), defaultdict(Counter)
    n_flower_imgs, rejected = 0, []
    for name, r in results:
        if not r["is_flower"]:
            rejected.append(name)
            continue
        n_flower_imgs += 1
        for it in r["items"]:
            total[it["label"]] += 1
            in_images[it["label"]].add(name)
            colors[it["label"]][it["color_name"]] += 1
    return {"total": total, "in_images": {k: len(v) for k, v in in_images.items()}, "colors": colors,
            "n_images": len(results), "n_flower_images": n_flower_imgs, "rejected": rejected,
            "n_flowers": sum(total.values())}


def draw_result(name, img, r):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.imshow(img)
    for i, it in enumerate(r["items"], 1):
        x0, y0, x1, y1 = it["box"]
        color = "orange" if it["label"] == UNKNOWN else "lime"
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, linewidth=2))
        ax.text(x0, max(y0 - 4, 8), f"{i}. {it['color_name']} {it['label']}{' (cluster)' if it.get('cluster') else ''}{' (fake)' if r.get('artificial') else ''} {it['conf']:.0%}", fontsize=8, color="black",
                bbox=dict(facecolor=color, alpha=0.85, pad=1))
    ax.set_title(name, fontsize=9)
    ax.axis("off")
    plt.show()


def run_all():
    if not image_bank:
        print("ยังไม่มีรูป — รันเซลล์ที่ 7 เพื่อเพิ่มรูปก่อน")
        return None
    results = []
    for k, (name, img) in enumerate(image_bank, 1):
        print("=" * 60)
        print(f"[{k}/{len(image_bank)}] กำลังตรวจ {name} ...", flush=True)
        r = analyze_image(img)
        results.append((name, r))
        if not r["is_flower"]:
            what, pct = r["looks_like"]
            display(HTML(
                "<div style='background:#fdecea;border:2px solid #d93025;color:#b3261e;"
                "padding:10px;border-radius:8px'>"
                + (f"⚠️ <b>{name}</b>: เป็นดอกไม้การ์ตูน/ในเกม/ภาพวาด ไม่นับ กรุณาใช้รูปถ่ายดอกไม้จริงหรือดอกไม้ปลอม<br>"
                   f"<span style='font-size:12px'>โอกาสเป็นการ์ตูน/เกม {pct:.1%}</span></div>"
                   if r.get("reason") == "drawn" else
                   f"⚠️ <b>{name}</b>: ไม่พบดอกไม้ในรูปนี้ กรุณาเพิ่มรูปดอกไม้เข้ามา<br>"
                   f"<span style='font-size:12px'>โอกาสเป็นดอกไม้ {r['p_flower']:.1%} "
                   f"(ใกล้เคียง: {what} {pct:.1%})</span></div>")))
            continue
        draw_result(name, img, r)
        c = Counter(it["label"] for it in r["items"])
        txt = ", ".join(f"{k} {v}" for k, v in c.most_common())
        note = "  (detector ไม่เจอกล่อง ใช้ทั้งรูปเป็น 1 ดอก)" if r["fallback"] else ""
        extra = f"  [ตัดกล่องที่ไม่ใช่ดอกไม้ {r['dropped']} กล่อง]" if r["dropped"] else ""
        print(f"{name}: พบ {len(r['items'])} ดอก → {txt}{note}{extra}")
        if r.get("artificial"):
            print(f"   🌼 ดอกไม้ปลอม ({r['p_fake']:.0%}) → ใช้ CLIP ช่วยบอกชนิด")
        if SHOW_TOP3:
            for i, it in enumerate(r["items"], 1):
                alts = ", ".join(f"{n} {p:.0%}" for n, p in it["top3"])
                print(f"   {i}. [{it['color_name']} RGB{it['color']}] {alts}")

    s = summarize(results)
    rows = "".join(
        f"<tr><td>{k}</td><td style='text-align:center'>{v}</td>"
        f"<td style='text-align:center'>{s['in_images'][k]}</td>"
        f"<td>{', '.join(f'{c} {n}' for c, n in s['colors'][k].most_common())}</td></tr>"
        for k, v in s["total"].most_common())
    rej = ""
    if s["rejected"]:
        rej = ("<p style='color:#b3261e'>⚠️ รูปที่ไม่ใช่ดอกไม้ (" + str(len(s["rejected"])) + " รูป): "
               + ", ".join(s["rejected"]) + " — กรุณาเพิ่มรูปดอกไม้</p>")
    display(HTML(
        "<div style='background:#e6f4ea;border:2px solid #188038;color:#0d3b1f;"
        "padding:14px;border-radius:8px;margin-top:12px'>"
        f"<h3 style='margin:0 0 8px 0'>🌸 สรุปรวม: {s['n_flowers']} ดอก จาก {s['n_flower_images']}/{s['n_images']} รูป</h3>"
        "<table style='border-collapse:collapse;min-width:320px'>"
        "<tr><th style='text-align:left'>ชนิด</th><th>จำนวนดอก</th><th>อยู่ในกี่รูป</th><th style='text-align:left'>สี</th></tr>"
        f"{rows}</table>{rej}</div>"))
    return results


results = run_all()

## 9) ส่งออกให้ขั้นเจนช่อดอกไม้
ได้ `results/bouquet_input/bouquet.json` + `cutouts/*.png` (รูปดอกตัดพื้นหลัง พื้นใส)
- `flowers` = รายการแบบเดียวกับเว็บ Bloom Craft: `{n: "Rose (Pink)", c: "#d98a98", q: 7}`
- `prompt` = ประโยคพร้อมใช้กับโมเดลเจนรูป (เปลี่ยน `STYLE` / `CONTAINER` ได้)
- `items` = รายดอก (ชนิด สี รหัสสี ขนาด ไฟล์ cutout) เผื่อใช้เป็นรูปอ้างอิง

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

EXPORT_DIR = "results/bouquet_input"
STYLE = "Soft & Romantic"         # ตามเว็บ: "Soft & Romantic" | "Wild Garden" | "Modern Minimal"
CONTAINER = "tall, narrow vase"   # ตามเว็บ: "hand-tied bouquet" | "tall, narrow vase" | "short, wide vase" | "mug"


def group_flowers(groups):
    """{(ชนิด, ชื่อสี): [rgb, ...]} → รายการดอกแบบเว็บ Bloom Craft + ข้อความสำหรับ prompt"""
    flowers, parts = [], []
    for (sp, col), rgbs in sorted(groups.items(), key=lambda g: -len(g[1])):
        med = np.median(np.array(rgbs), axis=0).astype(int)   # รหัสสีกลางของกลุ่ม
        name = "flower" if sp == UNKNOWN else sp
        flowers.append({"n": f"{name.title()} ({col.title()})", "c": "#%02x%02x%02x" % tuple(med), "q": len(rgbs)})
        plural = name if len(rgbs) == 1 else (name[:-1] + "ies" if name.endswith("y") else name + "s")
        parts.append(f"{len(rgbs)} {col} {plural}")
    return flowers, parts


def export_for_generation(results):
    os.makedirs(f"{EXPORT_DIR}/cutouts", exist_ok=True)
    imgs = dict(image_bank)
    items, groups = [], defaultdict(list)
    for name, r in results or []:
        if not r["is_flower"]:
            continue
        img = imgs[name]
        W, H = img.size
        for i, it in enumerate(r["items"], 1):
            x0, y0, x1, y1 = it["box"]
            cut = None
            if it.get("mask") is not None:   # cutout พื้นใสจาก mask ของ SAM 3
                c = img.crop((x0, y0, x1, y1)).convert("RGBA")
                c.putalpha(Image.fromarray(it["mask"].astype("uint8") * 255).resize(c.size))
                cut = f"cutouts/{Path(name).stem}_{i:02d}_{it['label']}.png"
                c.save(f"{EXPORT_DIR}/{cut}")
            items.append({"image": name, "species": it["label"], "color": it["color_name"],
                          "hex": "#%02x%02x%02x" % tuple(it["color"]),
                          "size": round((x1 - x0) * (y1 - y0) / (W * H), 4), "cutout": cut})
            groups[(it["label"], it["color_name"])].append(it["color"])
    flowers, parts = group_flowers(groups)
    prompt = (f"A {STYLE.lower()} flower arrangement in a {CONTAINER}, made of exactly "
              + ", ".join(parts) + ". Photorealistic, soft natural light, plain background.")
    out = {"flowers": flowers, "style": STYLE, "container": CONTAINER, "prompt": prompt, "items": items}
    json.dump(out, open(f"{EXPORT_DIR}/bouquet.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("บันทึก", f"{EXPORT_DIR}/bouquet.json", "|", len(items), "ดอก,", sum(i["cutout"] is not None for i in items), "cutout")
    print("prompt:", prompt)
    return out


bouquet = export_for_generation(results)

## 9.5) เจนรูปช่อดอกไม้ (ฟรี รันบน GPU ของ Colab)
**Realistic Vision (SD 1.5) + IP-Adapter Plus**: ใช้ prompt จากรายการดอก + **รูปดอกจริงที่ SAM 3 ตัดไว้เป็นตัวอ้างอิง** → ช่อที่ได้มีดอกหน้าตา/สีใกล้ดอกในรูปที่ถ่าย
- ครั้งแรกดาวน์โหลดโมเดลราว 4–5GB · ใช้ GPU ราว 3.5GB · เครื่องที่ไม่มี GPU ใหญ่จะข้ามเซลล์นี้
- `REF_SCALE` มาก = ดอกเหมือนรูปจริงมาก แต่ทำตาม prompt (แจกัน/สไตล์) น้อยลง
- โมเดลเจนรูปนับจำนวนดอกเป๊ะไม่ได้ จำนวนใน prompt เป็นแนวทาง


In [ ]:
%pip install -q diffusers accelerate
# ทางเลือก: SDXL สวยกว่าแต่ใหญ่ 8GB ทำแรม Colab ฟรีเต็มจนแครช (เก็บไว้ที่ old_versions/..._v5a_sdxl_ram_crash.ipynb)
import re, time, threading
import torch
from PIL import Image, ImageOps
from IPython.display import display

GEN_MODEL = "SG161222/Realistic_Vision_V5.1_noVAE"   # SD 1.5 แนวภาพถ่ายจริง (SDXL ใหญ่เกิน แรม Colab เต็มจนแครช)
GEN_STEPS = 30
GEN_W, GEN_H = 640, 768   # แนวตั้ง เหมาะกับช่อในแจกัน · SD 1.5 ใหญ่กว่านี้มักได้แจกันซ้อนหลายใบ
GEN_CANDIDATES = 3   # ช่อหลายชนิด: เจนได้สูงสุด 3 รูป เลือกรูปที่มีดอกครบทุกชนิด (SD 1.5 บางทีลืมไปหนึ่งชนิด)
FOUND_THR = 0.24     # CLIP ViT-B/32: มีดอกชนิดนั้นจริง ≈ 0.25+, ไม่มี/ผสมเป็นดอกเดียว ≈ 0.21
REF_SCALE_ZONE = 0.30  # หลายชนิด: ปรับลดเพื่อไม่ให้ดึงลักษณะดอกเดี่ยวยักษ์มาบวมทับ
REFINE_STRENGTH = 0.45  # รอบวาดซ้ำทีละคลัสเตอร์: คมชัดขึ้น แซมกันกลมกลืน ไม่ทำลายทรงช่อรวมและแจกัน
REF_SCALE = 0.25        # พอเหมาะสำหรับดอกชนิดเดียว
# prompt สั้นๆ: ตัวเข้ารหัสข้อความของ SD อ่านได้แค่ 77 token ส่วนเกินถูกตัดทิ้ง → ภาชนะต้องมาก่อน
STYLE_HINT = {   # ตามเว็บ Bloom Craft
    "Soft & Romantic": "soft romantic pastel style, airy garden feel",
    "Wild Garden": "wild garden style, loose natural freshly picked look",
    "Modern Minimal": "modern minimal style, sculptural lines, negative space",
}
CONTAINER_TXT = {   # ภาชนะ → (ประโยคขึ้นต้น prompt, สิ่งที่ห้ามมี)
    "hand-tied bouquet": ("A single hand-tied bouquet held in hands, stems wrapped in kraft paper with a ribbon",
                          "vase, glass, jar, pot, mug, cup, table, two bouquets, multiple bouquets, side by side, split image"),
    "tall, narrow vase": ("A single tall narrow clear glass vase holding an intertwined flower arrangement",
                          "hands, wrapping paper, mug, bowl, wide vase, two vases, multiple vases, double vase, side by side, split image"),
    "short, wide vase": ("A single short wide round ceramic bowl vase holding an intertwined flower arrangement",
                         "tall vase, hands, wrapping paper, mug, two vases, multiple vases, double vase, side by side, split image"),
    "small, round mug": ("A single small round ceramic coffee mug holding an intertwined flower arrangement",
                         "vase, glass vase, hands, wrapping paper, two mugs, side by side, split image"),
}
CONTAINERS = list(CONTAINER_TXT)
NEGATIVE = ("cartoon, drawing, illustration, painting, blurry, deformed petals, text, watermark, logo, "
            "flower field, garden background, hybrid flower, blended flower, two vases, multiple vases, "
            "double vase, side by side, split image, split view, divided screen, half and half, two bouquets, "
            "multiple bouquets, macro close-up, giant bloom, blurry blob")
LOOK = {   # บอกหน้าตาแต่ละชนิด กันโมเดลเอาหลายชนิดมาผสมเป็นดอกเดียว (จุดอ่อนของ SD 1.5)
    "sunflower": "with big dark brown centers",
    "daisy": "with yellow centers",
    "tulip": "with smooth cup-shaped petals",
    "rose": "with layered spiral petals",
}
_gpu = threading.Lock()   # ใช้ร่วมกับเซิร์ฟเวอร์ข้อ 10: ทีละงาน กัน GPU แรมเต็ม

gen_pipe = None
_vram = torch.cuda.get_device_properties(0).total_memory / 2**30 if device == "cuda" else 0
if GEN_URL:   # เจนที่เครื่อง SDXL แยก → ไม่ต้องโหลด SD 1.5 ประหยัดแรม
    print("เจนรูปที่เครื่องแยก:", GEN_URL)
elif _vram >= 14 and torch.cuda.mem_get_info()[0] / 2**30 >= 4.5:   # T4 = 15GB · ต้องเหลือว่างราว 4.5GB หลังโหลด SAM 3
    from diffusers import AutoencoderKL, DPMSolverMultistepScheduler, StableDiffusionPipeline
    from transformers import CLIPVisionModelWithProjection
    gen_pipe = StableDiffusionPipeline.from_pretrained(
        GEN_MODEL, torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False,
        vae=AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse", torch_dtype=torch.float16),
        image_encoder=CLIPVisionModelWithProjection.from_pretrained(
            "h94/IP-Adapter", subfolder="models/image_encoder", torch_dtype=torch.float16))
    gen_pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        gen_pipe.scheduler.config, algorithm_type="dpmsolver++", use_karras_sigmas=True)   # config เดิมของโมเดลเป็น deis ใช้คู่กันไม่ได้
    gen_pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter-plus_sd15.safetensors")
    gen_pipe.set_progress_bar_config(disable=True)
    gen_pipe.to("cuda")   # ทั้งตัวราว 3.5GB อยู่บน GPU ได้ ไม่กินแรมเครื่อง
    print(f"โมเดลเจนรูปพร้อม: {GEN_MODEL} + IP-Adapter Plus · GPU ว่างเหลือ {torch.cuda.mem_get_info()[0] / 2**30:.1f}GB")
else:
    print(f"ข้ามการเจนรูป: GPU ไม่พอ (ต้องเป็น Colab T4 ขึ้นไป และว่างอย่างน้อย 4.5GB)")


def _plural(name, q):
    return name if q == 1 else (name[:-1] + "ies" if name.endswith("y") else name + "s")


def bouquet_prompt(flowers, style, container):
    """flowers แบบเว็บ [{n: "Tulip (Pink)", q: 12}, ...] → prompt ภาษาอังกฤษ"""
    parts = []
    for f in flowers:
        m = re.match(r"^(.+?) \((.+)\)$", f["n"])
        name, col = (m.group(1), m.group(2) + " ") if m else (f["n"], "")
        look = f" {LOOK[name.lower()]}" if name.lower() in LOOK and len(flowers) <= 3 else ""   # เกิน 3 ชนิด prompt ยาวเกิน 77 token
        parts.append(f"{int(f['q'])} {col.lower()}{_plural(name.lower(), int(f['q']))}{look}")
    lead = CONTAINER_TXT.get(container, (f"A single flower arrangement in a {container}", ""))[0]
    return (f"{lead}, made of " + ", ".join(parts) + f". {STYLE_HINT.get(style, style)}. "
            "In a single cohesive arrangement, intertwining blooms and stems, boutique florist photo, plain light background, soft natural light, photorealistic")


def make_ref(pairs, max_n=4, cell=256):
    """[(รูป, items)] → [รูปดอกที่ตัดพื้นหลังบนพื้นขาว ชนิดละ 1 รูป (ดอกใหญ่สุด)] | None ถ้าไม่มี mask
    แยกรูปต่อชนิด: ถ้ารวมเป็นแผ่นเดียว IP-Adapter มองเป็นความหมายก้อนเดียว → ทิวลิป+ทานตะวันผสมเป็นดอกเดียว"""
    best = {}
    for img, items in pairs:
        for it in items:
            if it.get("mask") is None:
                continue
            x0, y0, x1, y1 = it["box"]
            k, a = it["label"], (x1 - x0) * (y1 - y0)
            if k not in best or a > best[k][0]:
                cut = img.crop((x0, y0, x1, y1)).convert("RGBA")
                cut.putalpha(Image.fromarray(it["mask"].astype("uint8") * 255).resize(cut.size))
                best[k] = (a, cut, it.get("color_name"))
    refs = []
    for k, (_, cut, col) in sorted(best.items(), key=lambda t: -t[1][0])[:max_n]:
        cut = ImageOps.contain(cut, (cell - 16, cell - 16))
        sq = Image.new("RGB", (cell, cell), "white")
        sq.paste(cut, ((cell - cut.width) // 2, (cell - cut.height) // 2), cut)
        sq.info.update(label=str(k).lower(), color=(col or "").lower())
        refs.append(sq)
    return refs or None


def zone_images(n, top=0.06, bottom=0.62):
    """สร้าง Mask แบบคลัสเตอร์ธรรมชาติ (Organic Clustered Ellipses) ซ้อนเหลื่อมกัน
    เพื่อให้ดอกไม้จัดแซมรวมกันเป็นพุ่มเดียวอย่างเป็นธรรมชาติ ไม่ผ่าครึ่งซ้าย-ขวา"""
    from PIL import ImageDraw
    ims = []
    cluster_configs = [
        [(0.42, 0.35, 0.32, 0.22), (0.58, 0.34, 0.32, 0.22)],
        [(0.40, 0.38, 0.28, 0.20), (0.60, 0.38, 0.28, 0.20), (0.50, 0.24, 0.32, 0.18)],
        [(0.38, 0.36, 0.26, 0.18), (0.62, 0.36, 0.26, 0.18), (0.50, 0.22, 0.28, 0.16), (0.50, 0.45, 0.26, 0.16)]
    ]
    configs = cluster_configs[min(max(0, n - 2), len(cluster_configs) - 1)]
    for i in range(n):
        m = Image.new("L", (GEN_W, GEN_H), 0)
        cx_rel, cy_rel, rx_rel, ry_rel = configs[i % len(configs)]
        cx, cy = int(GEN_W * cx_rel), int(GEN_H * cy_rel)
        rx, ry = int(GEN_W * rx_rel), int(GEN_H * ry_rel)
        ImageDraw.Draw(m).ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=255)
        ims.append(m)
    return ims


def zone_masks(n):
    from diffusers.image_processor import IPAdapterMaskProcessor
    ims = zone_images(n)
    t = IPAdapterMaskProcessor().preprocess(ims, height=GEN_H, width=GEN_W)   # [n,1,H,W]
    return [t.reshape(1, n, t.shape[2], t.shape[3])]


inpaint_pipe = None


def refine_zones(im, refs, container, seed):
    """วาดซ้ำทีละคลัสเตอร์ด้วย prompt ที่มีแค่ชนิดของโซนนั้น (ไม่ใส่ lead เพื่อไม่ให้แจกันหรือกระดาษห่องอกซ้ำ)"""
    global inpaint_pipe
    if inpaint_pipe is None:
        from diffusers import StableDiffusionInpaintPipeline
        inpaint_pipe = StableDiffusionInpaintPipeline(**gen_pipe.components)   # ใช้โมเดลชุดเดิม ไม่กิน GPU เพิ่ม
        inpaint_pipe.set_progress_bar_config(disable=True)
    labels = [r.info.get("label", "") for r in refs]
    for i, (r, z) in enumerate(zip(refs, zone_images(len(refs)))):
        lab, col = labels[i], r.info.get("color", "")
        if not lab:
            continue
        name = _plural(lab, 2)
        others = ", ".join(_plural(l, 2) for l in labels if l and l != lab)
        gen_pipe.set_ip_adapter_scale(0.28)
        im = inpaint_pipe(
            prompt=f"fresh blooming {col} {name} {LOOK.get(lab, '')}, distinct petals, naturally intertwined among other flowers, sharp focus, florist arrangement, photorealistic".replace("  ", " "),
            negative_prompt=NEGATIVE + ", vase, vessel, glass, pot, jar, mug, bowl, hands, wrapping paper, stems cut, two vases, multiple vases, side by side, split image, macro, giant bloom, " + others,
            image=im, mask_image=inpaint_pipe.mask_processor.blur(z, 36),
            ip_adapter_image=r, strength=REFINE_STRENGTH, num_inference_steps=GEN_STEPS, guidance_scale=6.0,
            width=GEN_W, height=GEN_H, generator=torch.Generator("cpu").manual_seed(seed + 100 + i)).images[0]
        print(f"   วาดซ้ำคลัสเตอร์ {i + 1}: {col} {name}", flush=True)
    return im


CHECK = {"rose": "roses", "tulip": "tulips", "sunflower": "sunflowers", "dais": "daisies"}   # คำใน prompt → ชื่อที่ให้ CLIP หา


@torch.inference_mode()
def species_coverage(img, prompt):
    """คะแนนของชนิดที่ 'เจอน้อยที่สุด' ในรูป (CLIP) · None ถ้า prompt มีชนิดเดียว"""
    names = [v for k, v in CHECK.items() if k in prompt.lower()]
    if len(names) < 2:
        return None
    t = encode_text(gate_model, gate_tok, [f"a photo of {n}" for n in names]).float()
    return (t @ encode_image(gate_model, gate_preprocess, img).float()).min().item()


def generate_bouquet(prompt, ref=None, seed=None, container=None):
    if gen_pipe is None:
        raise RuntimeError("ไม่มีโมเดลเจนรูป (ต้องใช้ GPU แบบ Colab T4)")
    refs = ref if isinstance(ref, list) else ([ref] if ref is not None else [])
    if len(refs) >= 2:   # หลายชนิด → แบ่งโซน ชนิดละโซน
        gen_pipe.set_ip_adapter_scale([[REF_SCALE_ZONE] * len(refs)])
        ip_img, xkw = [refs], {"ip_adapter_masks": zone_masks(len(refs))}
    else:   # ชนิดเดียว / ไม่มีรูปอ้างอิง (scale 0 → ใช้ prompt อย่างเดียว)
        gen_pipe.set_ip_adapter_scale(REF_SCALE if refs else 0.0)
        ip_img, xkw = (refs[0] if refs else Image.new("RGB", (224, 224), "white")), None
    seed = int(time.time()) if seed is None else seed
    neg = NEGATIVE + (", " + CONTAINER_TXT[container][1] if container in CONTAINER_TXT else "")   # ห้ามภาชนะอื่น
    best = None
    with _gpu, torch.inference_mode():
        for k in range(GEN_CANDIDATES):
            im = gen_pipe(prompt=prompt, negative_prompt=neg, ip_adapter_image=ip_img, cross_attention_kwargs=xkw,
                          num_inference_steps=GEN_STEPS, guidance_scale=6.0, width=GEN_W, height=GEN_H,
                          generator=torch.Generator("cpu").manual_seed(seed + k)).images[0]
            score = species_coverage(im, prompt)
            if score is None:   # ชนิดเดียว ไม่ต้องคัด
                return im
            print(f"   รูปที่ {k + 1}: คะแนนชนิดที่เจอน้อยสุด {score:.3f}", flush=True)
            if best is None or score > best[0]:
                best = (score, im)
            if score >= FOUND_THR:   # ครบทุกชนิดแล้ว ไม่ต้องเจนต่อ
                break
        return refine_zones(best[1], refs, container, seed) if len(refs) >= 2 else best[1]


# ลองเจนจากรูปเดโมทุกรูป (ผลจากข้อ 8-9) → results/bouquet_input/generated.png
if gen_pipe is not None and results:
    imgs = dict(image_bank)
    demo_ref = make_ref([(imgs[n], r["items"]) for n, r in results if r["is_flower"]])
    demo_prompt = bouquet_prompt(bouquet["flowers"][:6], STYLE, CONTAINER)
    t0 = time.time()
    out = generate_bouquet(demo_prompt, demo_ref, seed=0, container=CONTAINER)
    out.save(f"{EXPORT_DIR}/generated.png")
    for i, r in enumerate(demo_ref or []):
        r.save(f"{EXPORT_DIR}/generated_ref_{i}.png")
    print(f"เจนเสร็จใน {time.time() - t0:.0f} วินาที → {EXPORT_DIR}/generated.png\nprompt:", demo_prompt)
    display(ImageOps.contain(out, (512, 512)))



## 10) เปิดเว็บ Bloom Craft ให้ส่งรูปมาวิเคราะห์ได้
รันเซลล์นี้แล้วจะได้ลิงก์ `https://….trycloudflare.com` → เปิดในมือถือหรือคอมเครื่องไหนก็ได้ → อัปโหลดได้ครั้งละหลายรูป (≤ 10 รวมเป็นช่อเดียว) หรือถ่ายรูป → **Use this crop** → เว็บโชว์กรอบ ชนิด สี จำนวนจริงจากโมเดล
- ลิงก์ใช้ได้เฉพาะตอนที่ Colab ยังเปิดอยู่ (ปิด Colab = ลิงก์ตาย รันใหม่ได้ลิงก์ใหม่)
- บนเครื่องตัวเองเปิด http://localhost:8000


In [ ]:
%pip install -q fastapi uvicorn python-multipart
import base64, io, re, subprocess, threading, time, uuid
from fastapi import Body, FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
from fastapi.staticfiles import StaticFiles
from PIL import Image, ImageDraw
import uvicorn

WEB_DIR = "เว็บ_Bloom_Craft"
PORT = 8000
MAX_BYTES = 12 * 1048576   # ต่อรูป เท่ากับที่เว็บจำกัดไว้
MAX_IMAGES = 10            # ต่อครั้ง (Cloudflare ตัดที่ 100 วินาที)
_refs = {}   # token → รูปอ้างอิงดอกจริงของการวิเคราะห์ครั้งนั้น (ใช้ตอนกด Generate)


def _data_url(im, quality=85):
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=quality)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def _annotated(img, items):
    im = img.copy()
    d = ImageDraw.Draw(im)
    w = max(2, im.width // 250)
    for i, it in enumerate(items, 1):
        col = "orange" if it["label"] == UNKNOWN else "lime"
        d.rectangle(it["box"], outline=col, width=w)
        d.text((it["box"][0] + w + 2, it["box"][1] + w + 2), f"{i}. {it['color_name']} {it['label']}", fill=col)
    im.thumbnail((900, 900))
    return _data_url(im)


if globals().get("_web_server") is None:
    api = FastAPI()
else:   # รันเซลล์ซ้ำ: ลงทะเบียน route ใหม่บนแอปที่เซิร์ฟเวอร์ใช้อยู่ → โค้ดใหม่มีผลทันที ไม่ต้อง restart
    api.router.routes.clear()


@api.post("/api/analyze")
def api_analyze(images: list[UploadFile] = File(...)):   # def ธรรมดา → FastAPI รันใน thread ไม่บล็อกเซิร์ฟเวอร์
    if len(images) > MAX_IMAGES:
        return JSONResponse({"error": "too_many"}, 413)
    pairs, photos, rejected = [], [], []
    for f in images:
        data = f.file.read(MAX_BYTES + 1)
        if len(data) > MAX_BYTES:
            return JSONResponse({"error": "too_big", "file": f.filename}, 413)
        try:
            img = Image.open(io.BytesIO(data)).convert("RGB")
        except Exception:
            return JSONResponse({"error": "bad_image", "file": f.filename}, 400)
        img.thumbnail((2048, 2048))   # รูปมือถือใหญ่มาก ย่อก่อน
        with _gpu:
            r = analyze_image(img)
        if not r["is_flower"]:
            rejected.append({"file": f.filename, "reason": r.get("reason")})
            continue
        pairs.append((img, r["items"]))
        photos.append(_annotated(img, r["items"]))
    if not pairs:
        return {"is_flower": False, "reason": rejected[0]["reason"] if rejected else None, "rejected": rejected}
    groups = defaultdict(list)   # รวมดอกจากทุกรูปเป็นช่อเดียว (เหมือน export_for_generation)
    for _, items in pairs:
        for it in items:
            groups[(it["label"], it["color_name"])].append(it["color"])
    flowers, parts = group_flowers(groups)
    n = sum(len(items) for _, items in pairs)
    print(f"[web] {len(pairs)}/{len(images)} รูป {n} ดอก → " + ", ".join(parts), flush=True)
    token = None
    ref = make_ref(pairs)
    if ref is not None:
        token = uuid.uuid4().hex
        _refs[token] = ref
        while len(_refs) > 20:   # เก็บแค่ 20 ชุดล่าสุด
            _refs.pop(next(iter(_refs)))
    return {"is_flower": True, "n": n, "flowers": flowers, "photos": photos, "rejected": rejected, "ref": token}


def _remote_generate(prompt, ref, container):
    """ส่งงานเจนไปเครื่อง SDXL แยก (bloom_craft_gen_sdxl.ipynb) พร้อมรูปอ้างอิงชนิดละรูป"""
    import requests
    refs = ref if isinstance(ref, list) else ([ref] if ref is not None else [])
    out = []
    for r in refs:
        lab, col = r.info.get("label", ""), r.info.get("color", "")
        buf = io.BytesIO()
        r.save(buf, "PNG")
        out.append({"png": base64.b64encode(buf.getvalue()).decode(), "plural": _plural(lab, 2) if lab else "",
                    "phrase": f"{col} {_plural(lab, 2)} {LOOK.get(lab, '')}".strip() if lab else ""})
    body = {"prompt": prompt, "refs": out,
            "negative": NEGATIVE + (", " + CONTAINER_TXT[container][1] if container in CONTAINER_TXT else ""),
            "lead": CONTAINER_TXT.get(container, (f"A flower arrangement in a {container}", ""))[0]}
    t0 = time.time()
    try:
        r = requests.post(GEN_URL.rstrip("/") + "/generate", json=body, timeout=95)   # Cloudflare ฝั่งเว็บตัดที่ 100 วินาที
        r.raise_for_status()
    except Exception as e:
        print("[web] เครื่องเจนรูปแยกไม่ตอบ:", e, flush=True)
        return JSONResponse({"error": "gen_server", "detail": str(e)[:200]}, 502)
    print(f"[web] เจนรูป (SDXL แยก) {time.time() - t0:.0f} วินาที | อ้างอิงดอกจริง {len(out)} รูป | {prompt}", flush=True)
    return {"image": r.json()["image"], "prompt": prompt, "used_ref": bool(out)}


@api.post("/api/generate")
def api_generate(req: dict = Body(...)):
    if gen_pipe is None and not GEN_URL:
        return JSONResponse({"error": "no_generator"}, 503)
    fl = [f for f in req.get("flowers") or [] if isinstance(f, dict) and isinstance(f.get("n"), str)
          and len(f["n"]) <= 60 and isinstance(f.get("q"), (int, float)) and 1 <= f["q"] <= 200][:6]   # เกิน 6 ชนิด prompt ยาวเกิน 77 token
    if not fl:
        return JSONResponse({"error": "no_flowers"}, 400)
    style = req.get("style") if req.get("style") in STYLE_HINT else STYLE
    container = req.get("container") if req.get("container") in CONTAINERS else CONTAINER
    prompt = bouquet_prompt([{"n": re.sub(r"[^\w ()&'-]", "", f["n"]), "q": int(f["q"])} for f in fl], style, container)
    ref = _refs.get(req.get("ref")) if isinstance(req.get("ref"), str) else None
    if isinstance(ref, list):   # ผู้ใช้ลบชนิดออกในเว็บ → ทิ้งรูปอ้างอิงของชนิดนั้นด้วย (ไม่งั้นยังได้โซนของมัน)
        keep = {re.sub(r" \(.*\)$", "", f["n"]).strip().lower() for f in fl}
        keep |= {UNKNOWN} if "flower" in keep else set()   # เว็บเรียกดอกไม่รู้ชนิดว่า "Flower"
        ref = [r for r in ref if r.info.get("label") in keep] or None
    if GEN_URL:
        return _remote_generate(prompt, ref, container)
    t0 = time.time()
    im = generate_bouquet(prompt, ref, container=container)
    print(f"[web] เจนรูป {time.time() - t0:.0f} วินาที | อ้างอิงดอกจริง: {'ใช่' if ref else 'ไม่'} | {prompt}", flush=True)
    return {"image": _data_url(im, 90), "prompt": prompt, "used_ref": ref is not None}


api.mount("/", StaticFiles(directory=WEB_DIR, html=True), name="web")

if globals().get("_web_server") is None:   # รันเซลล์ซ้ำ → ไม่เปิดพอร์ตซ้ำ
    _web_server = uvicorn.Server(uvicorn.Config(api, host="0.0.0.0", port=PORT, log_level="warning"))
    threading.Thread(target=_web_server.run, daemon=True).start()
    time.sleep(2)

if "google.colab" in sys.modules:
    if not os.path.exists("cloudflared"):
        subprocess.run(["wget", "-q", "-O", "cloudflared",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod("cloudflared", 0o755)
    if globals().get("_tunnel_url") is None or _tun.poll() is not None:   # tunnel ตาย → เปิดใหม่ ได้ลิงก์ใหม่
        _tunnel_url = None
        _tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
                                stderr=subprocess.PIPE, text=True)
        t0 = time.time()
        for line in _tun.stderr:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m:
                _tunnel_url = m.group(0)
                break
            if time.time() - t0 > 60:
                break
        threading.Thread(target=lambda: [None for _ in _tun.stderr], daemon=True).start()   # อ่านทิ้ง กัน pipe เต็ม
    if globals().get("_tunnel_url"):
        print("🌸 เปิดเว็บที่:", _tunnel_url)
    else:
        from google.colab.output import eval_js
        print("⚠️ เปิด tunnel ไม่ได้ ใช้ลิงก์สำรอง (เปิดได้เฉพาะเบราว์เซอร์ที่ล็อกอิน Colab นี้):",
              eval_js(f"google.colab.kernel.proxyPort({PORT})"))
else:
    print(f"🌸 เปิดเว็บที่: http://localhost:{PORT}")
